# Data Loading
**Goal:** Loading both raw CSV files, understand their structure, and document what we're working with before any cleaning or transformation.

> **Rule:** This notebook does NOT modify the data. We only read and observe.  
> Cleaning happens in `data_cleaning.ipynb`.

---
**Datasets:**
- `flights.csv` — 271,888 flight booking records. Target variable = `price`
- `users.csv`   — 1,340 user profiles. Adds `age`, `gender`, `company` as features
- `hotels.csv` is excluded — it belongs to the hotel recommendation model.

# Imports and Path Setup

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

# Works whether I launch jupyter from the project root OR from notebooks/
BASE_DIR = Path(os.getcwd())
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"

print(f"Project root : {BASE_DIR}")
print(f"Raw data     : {RAW_DIR}")
print(f"flights.csv exists : {(RAW_DIR / 'flights.csv').exists()}")
print(f"users.csv   exists : {(RAW_DIR / 'users.csv').exists()}")

Project root : c:\Voyage_Project
Raw data     : c:\Voyage_Project\data\raw
flights.csv exists : True
users.csv   exists : True


# Loading Raw Data

In [2]:
flights = pd.read_csv(RAW_DIR / "flights.csv")
users   = pd.read_csv(RAW_DIR / "users.csv")

print("=" * 45)
print(f"  flights.csv {flights.shape[0]:,} rows  ×  {flights.shape[1]} columns")
print(f"  users.csv   {users.shape[0]:,} rows  ×  {users.shape[1]} columns")
print("=" * 45)

  flights.csv 271,888 rows  ×  10 columns
  users.csv   1,340 rows  ×  5 columns


## 3. Flights Dataset - First Look
**Columns:**
| Column | Meaning |
|---|---|
| `travelCode` | Unique trip ID |
| `userCode` | Links to users.csv |
| `from` | Origin city |
| `to` | Destination city |
| `flightType` | economic / premium / firstClass |
| `price` | **Target variable** — what we want to predict |
| `time` | Flight duration (hours) |
| `distance` | Route distance (km) |
| `agency` | Booking agency |
| `date` | Booking date |

In [3]:
print("── First 5 rows ──")
display(flights.head())

print("\n── Last 5 rows ──")
display(flights.tail())

── First 5 rows ──


,travelCode,userCode,from,to,flightType,price,time,distance,agency,date
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,09/26/2019
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,09/30/2019
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,10/03/2019
3,1,0,Florianopolis (SC),Brasilia (DF),firstClass,1127.36,1.66,637.56,CloudFy,10/04/2019
4,2,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,10/10/2019



── Last 5 rows ──


,travelCode,userCode,from,to,flightType,price,time,distance,agency,date
271883,135941,1339,Campo Grande (MS),Florianopolis (SC),firstClass,1446.34,1.49,573.81,CloudFy,07/12/2020
271884,135942,1339,Florianopolis (SC),Natal (RN),economic,726.95,1.84,709.37,CloudFy,07/16/2020
271885,135942,1339,Natal (RN),Florianopolis (SC),economic,873.07,1.84,709.37,CloudFy,07/20/2020
271886,135943,1339,Florianopolis (SC),Rio de Janeiro (RJ),economic,313.62,1.21,466.30,CloudFy,07/23/2020
271887,135943,1339,Rio de Janeiro (RJ),Florianopolis (SC),economic,533.69,1.21,466.30,CloudFy,07/26/2020


### 3a. Column Data Types

In [4]:
print("── Data Types ──")
print(flights.dtypes)
print()

# Quick check — which columns are text (object) and need encoding later
text_cols = flights.select_dtypes(include="object").columns.tolist()
num_cols  = flights.select_dtypes(include=[np.number]).columns.tolist()

print(f"Text columns  (need encoding)  : {text_cols}")
print(f"Numeric columns                : {num_cols}")

── Data Types ──
travelCode      int64
userCode        int64
from              str
to                str
flightType        str
price         float64
time          float64
distance      float64
agency            str
date              str
dtype: object

Text columns  (need encoding)  : ['from', 'to', 'flightType', 'agency', 'date']
Numeric columns                : ['travelCode', 'userCode', 'price', 'time', 'distance']


### 3b. Basic Statistics (Numeric Columns)

In [5]:
display(flights.describe().round(2))

,travelCode,userCode,price,time,distance
count,271888.00,271888.00,271888.00,271888.00,271888.00
mean,67971.50,667.51,957.38,1.42,546.96
std,39243.72,389.52,362.31,0.54,208.85
min,0.00,0.00,301.51,0.44,168.22
25%,33985.75,326.00,672.66,1.04,401.66
50%,67971.50,659.00,904.00,1.46,562.14
75%,101957.25,1011.00,1222.24,1.76,676.53
max,135943.00,1339.00,1754.17,2.44,937.77


### Basic Statistics (Numeric Columns)

- The dataset contains **271,888 flight records** with an average ticket price of **957.38** and an average travel time of **1.42 hours**.
- Flight distances range from **168.22 km to 937.77 km**, with an average distance of **546.96 km**.

### 3c. Categorical Columns (Unique values)

In [6]:
for col in ["flightType", "agency"]:
    print(f"── {col} ──")
    print(flights[col].value_counts())
    print()

── flightType ──
flightType
firstClass    116418
premium        78004
economic       77466
Name: count, dtype: int64

── agency ──
agency
Rainbow        116752
CloudFy        116378
FlyingDrops     38758
Name: count, dtype: int64



### Categorical Columns Analysis

- **First Class** is the most common flight type (**116,418 bookings**), followed by **Premium (78,004)** and **Economic (77,466)**.
- Among agencies, **Rainbow (116,752 bookings)** and **CloudFy (116,378 bookings)** dominate, while **FlyingDrops (38,758 bookings)** handles fewer bookings.

### 3d. Date Range

In [7]:
flights["date"] = pd.to_datetime(flights["date"], format="%m/%d/%Y")

print(f"Date range  : {flights['date'].min().date()}  →  {flights['date'].max().date()}")
print(f"Total months: {flights['date'].dt.to_period('M').nunique()}")
print()
print("Bookings per year:")
print(flights["date"].dt.year.value_counts().sort_index())

Date range  : 2019-09-26  →  2023-07-24
Total months: 47

Bookings per year:
date
2019     35826
2020    112571
2021     75363
2022     41761
2023      6367
Name: count, dtype: int64


### Date Range Analysis

- The dataset contains booking records from **26-Sep-2019 to 24-Jul-2023**, covering **47 months** of data.
- The yearly booking count shows that **2020 had the highest number of bookings (112,571)**, while **2023 has fewer bookings because it contains only partial-year data**.

## 4. Users Dataset — First Look
**Columns:**
| Column | Meaning |
|---|---|
| `code` | Unique user ID — joins with flights.userCode |
| `company` | Employer of the traveller |
| `name` | Traveller name (will be dropped before model) |
| `gender` | male / female / none |
| `age` | Age in years |

In [8]:
print("── First 5 rows ──")
display(users.head())

print("\n── Last 5 rows ──")
display(users.tail())

── First 5 rows ──


,code,company,name,gender,age
0,0,4You,Roy Braun,male,21
1,1,4You,Joseph Holsten,male,37
2,2,4You,Wilma Mcinnis,female,48
3,3,4You,Paula Daniel,female,23
4,4,4You,Patricia Carson,female,44



── Last 5 rows ──


,code,company,name,gender,age
1335,1335,Umbrella LTDA,Albert Garroutte,male,23
1336,1336,Umbrella LTDA,Kim Shores,female,40
1337,1337,Umbrella LTDA,James Gimenez,male,28
1338,1338,Umbrella LTDA,Viola Agosta,female,52
1339,1339,Umbrella LTDA,Paul Rodriguez,male,35


### 4a. Column Data Types

In [9]:
print(users.dtypes)
print()

text_cols_u = users.select_dtypes(include="object").columns.tolist()
num_cols_u  = users.select_dtypes(include=[np.number]).columns.tolist()

print(f"Text columns  : {text_cols_u}")
print(f"Numeric columns: {num_cols_u}")

code       int64
company      str
name         str
gender       str
age        int64
dtype: object

Text columns  : ['company', 'name', 'gender']
Numeric columns: ['code', 'age']


### 4b. Basic Statistics

In [10]:
display(users.describe().round(2))

,code,age
count,1340.00,1340.00
mean,669.50,42.74
std,386.97,12.87
min,0.00,21.00
25%,334.75,32.00
50%,669.50,42.00
75%,1004.25,54.00
max,1339.00,65.00


### Users Dataset Summary

- The dataset contains **1,340 users** with an average age of **42.74 years**.
- User ages range from **21 to 65 years**, with **50% of users aged between 32 and 54 years** (median age = 42).

### 4c. Categorical Columns - unique values

In [11]:
for col in ["gender", "company"]:
    print(f"── {col} ──")
    print(users[col].value_counts())
    print()

── gender ──
gender
male      452
female    448
none      440
Name: count, dtype: int64

── company ──
company
4You             453
Acme Factory     261
Wonka Company    237
Monsters CYA     195
Umbrella LTDA    194
Name: count, dtype: int64



### Categorical Feature Analysis

- The gender distribution is fairly balanced, with **452 males**, **448 females**, and **440 users with unspecified gender**.
- Among companies, **4You** has the highest number of users (**453**), followed by **Acme Factory (261)** and **Wonka Company (237)**.

# 5. Quick Merge Preview

In [12]:
merged = flights.merge(
    users.rename(columns={"code": "userCode"}),
    on="userCode",
    how="left"
)

print(f"After merge  →  {merged.shape[0]:,} rows  ×  {merged.shape[1]} columns")
print(f"New columns added from users: {[c for c in merged.columns if c not in flights.columns]}")
print()
display(merged.head(3))

After merge  →  271,888 rows  ×  14 columns
New columns added from users: ['company', 'name', 'gender', 'age']



,travelCode,userCode,from,to,flightType,price,time,distance,agency,date,company,name,gender,age
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,2019-09-26,4You,Roy Braun,male,21
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,2019-09-30,4You,Roy Braun,male,21
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,2019-10-03,4You,Roy Braun,male,21


### Merge Results

- The merge produced **271,888 records** and **14 columns**, successfully combining flight and user information.
- Four new user-related columns were added: **company**, **name**, **gender**, and **age**.